# SREG — Exploración paso a paso

Esta notebook te permite ver exactamente qué hace cada parte del sistema.

**Secciones:**
1. Crear un mundo (la red bayesiana)
2. Visualizar la estructura causal
3. Samplear la "verdad oculta"
4. Jugar un episodio paso a paso
5. Puntuar al agente
6. Comparar teacher vs agente random

In [ ]:
# Setup
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import networkx as nx

from sreg.tools.world_gen import WorldGenTool, WorldGenConfig
from sreg.tools.world_check import WorldCheckTool
from sreg.solver.exact_bayes import ExactBayesSolver
from sreg.tools.episode_gen import EpisodeGenTool, EpisodeGenConfig
from sreg.tools.task_gen import TaskGenTool
from sreg.models.task import TaskSpec, TaskType
from sreg.models.world import NodeType
from sreg.env.episode import EpisodeRunner
from sreg.models.episode import Action, ActionType
from sreg.tools.verifier import VerifierTool

# Estilo global para graficos
plt.rcParams.update({
    "figure.facecolor": "#fafafa",
    "axes.facecolor": "#fafafa",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## 1. Crear un mundo

Un "mundo" es una red bayesiana: nodos conectados por flechas causales, cada uno con probabilidades condicionales.

**Parámetros que podés tocar:**
- `seed` — cambia esto para generar mundos distintos
- `num_nodes` — cuántos nodos en total (3-20)
- `edge_strength` — qué tan fuerte es la relación causa-efecto (0.1=ruidoso/difícil, 1.0=determinístico/fácil)
- `num_states` — cuántos estados tiene cada nodo (2=binario, 3=low/medium/high, etc.)

In [ ]:
# ===== CAMBIA ESTOS PARAMETROS PARA EXPLORAR =====
SEED = 5
NUM_NODES = 6
EDGE_STRENGTH = 0.7   # 0.1 = dificil, 0.7 = facil, 1.0 = casi deterministico
NUM_STATES = 3         # 2 = binario, 3 = low/med/high
EPISODE_SEED = 0
BUDGET = 4
# ==================================================

gen = WorldGenTool()
config = WorldGenConfig(
    seed=SEED,
    num_nodes=NUM_NODES,
    edge_strength=EDGE_STRENGTH,
    num_states=NUM_STATES,
)
world = gen.generate(config)

print(f"Mundo: {world.id}  |  Dificultad: {world.difficulty.level}  |  Template: {world.template_family}")

df_nodes = pd.DataFrame([
    {"Nodo": n.name, "Tipo": n.type.value, "Estados": ", ".join(n.states)}
    for n in world.nodes
])
display(df_nodes)

df_edges = pd.DataFrame([
    {"Desde": e.from_node, "Hacia": e.to_node}
    for e in world.edges
])
display(df_edges)

## 2. Visualizar la estructura causal

Dibujamos el grafo. Los colores indican el tipo de nodo:
- 🔴 Rojo = **latente** (oculto, el agente no puede ver esto)
- 🟢 Verde = **observable** (el agente puede elegir mirarlo, gasta presupuesto)
- 🟡 Amarillo = **target** (lo que hay que predecir)

In [ ]:
# Construir grafo de networkx
dag = nx.DiGraph()
for node in world.nodes:
    dag.add_node(node.name, type=node.type, states=node.states)
for edge in world.edges:
    dag.add_edge(edge.from_node, edge.to_node)

# Colores y estilos por tipo
palette = {
    "latent":     {"fill": "#ff6b6b", "edge": "#c92a2a", "text": "#fff"},
    "observable": {"fill": "#51cf66", "edge": "#2b8a3e", "text": "#fff"},
    "target":     {"fill": "#ffd43b", "edge": "#e67700", "text": "#1a1a2e"},
}

node_colors = [palette[dag.nodes[n]["type"]]["fill"] for n in dag.nodes]
edge_colors = [palette[dag.nodes[n]["type"]]["edge"] for n in dag.nodes]

# Layout jerárquico: nodos raíz arriba, hojas abajo
try:
    # Intento layout jerárquico con graphviz si está disponible
    from networkx.drawing.nx_agraph import graphviz_layout
    pos = graphviz_layout(dag, prog="dot")
except Exception:
    # Fallback: layout topológico manual
    topo = list(nx.topological_sort(dag))
    layers = {}
    for n in topo:
        preds = list(dag.predecessors(n))
        layer = max((layers[p] for p in preds), default=-1) + 1
        layers[n] = layer
    # Agrupar por layer
    by_layer = {}
    for n, l in layers.items():
        by_layer.setdefault(l, []).append(n)
    max_layer = max(by_layer.keys())
    pos = {}
    for l, nodes in by_layer.items():
        for i, n in enumerate(nodes):
            x = (i - (len(nodes) - 1) / 2) * 2.0
            y = (max_layer - l) * 2.0  # raíces arriba
            pos[n] = (x, y)

fig, ax = plt.subplots(figsize=(11, 8), facecolor="#fafafa")
ax.set_facecolor("#fafafa")

# Dibujar edges con estilo curvo
nx.draw_networkx_edges(
    dag, pos, ax=ax,
    edge_color="#adb5bd",
    width=2.0,
    arrows=True,
    arrowsize=25,
    arrowstyle="-|>",
    connectionstyle="arc3,rad=0.12",
    min_source_margin=25,
    min_target_margin=25,
)

# Dibujar nodos como círculos con sombra
for n in dag.nodes:
    x, y = pos[n]
    ntype = dag.nodes[n]["type"]
    p = palette[ntype]
    # Sombra
    shadow = plt.Circle((x + 0.06, y - 0.06), 0.55, color="#00000015", zorder=1)
    ax.add_patch(shadow)
    # Nodo
    circle = plt.Circle((x, y), 0.55, color=p["fill"], ec=p["edge"], linewidth=2.5, zorder=2)
    ax.add_patch(circle)
    # Nombre del nodo
    label = n.replace("_", "\n")
    ax.text(x, y + 0.08, label, ha="center", va="center", fontsize=8, fontweight="bold",
            color=p["text"], zorder=3,
            path_effects=[pe.withStroke(linewidth=2, foreground=p["fill"])])
    # Estados debajo
    states = dag.nodes[n].get("states", [])
    if states:
        states_text = " | ".join(states)
        ax.text(x, y - 0.35, states_text, ha="center", va="center", fontsize=6.5,
                color=p["text"], alpha=0.85, zorder=3)

# Leyenda
legend_items = [
    mpatches.Patch(facecolor="#ff6b6b", edgecolor="#c92a2a", linewidth=1.5, label="🔒 Latente (oculto)"),
    mpatches.Patch(facecolor="#51cf66", edgecolor="#2b8a3e", linewidth=1.5, label="👁️ Observable"),
    mpatches.Patch(facecolor="#ffd43b", edgecolor="#e67700", linewidth=1.5, label="🎯 Target (predecir)"),
]
ax.legend(handles=legend_items, loc="upper left", fontsize=10, framealpha=0.9,
          edgecolor="#dee2e6", fancybox=True, shadow=True)

ax.set_title(f"Estructura Causal — {world.id}", fontsize=15, fontweight="bold",
             color="#1a1a2e", pad=15)
ax.set_aspect("equal")
ax.axis("off")

# Ajustar márgenes
all_x = [p[0] for p in pos.values()]
all_y = [p[1] for p in pos.values()]
margin = 1.5
ax.set_xlim(min(all_x) - margin, max(all_x) + margin)
ax.set_ylim(min(all_y) - margin, max(all_y) + margin)

plt.tight_layout()
plt.show()

## 3. Validar el mundo

El `WorldCheckTool` revisa que el mundo sea interesante y válido:
- ¿Es un DAG? (no tiene ciclos)
- ¿Tiene nodos ocultos?
- ¿Los observables están conectados al target?
- ¿La entropía del target es suficiente? (que no sea trivial de adivinar)
- ¿Hay d-separaciones? (independencias condicionales interesantes)

In [ ]:
checker = WorldCheckTool()
result = checker.check(world)

print("VALIDACION:", "PASO" if result.passed else "FALLO")
if not result.passed:
    for f in result.failures:
        print(f"  - {f}")

df_metrics = pd.DataFrame(
    [{"Metrica": k, "Valor": round(v, 3)} for k, v in result.metrics.items()]
)
display(df_metrics)

## 4. Samplear la "verdad oculta"

Ahora "tiramos los dados". El sistema recorre los nodos en orden causal y samplea un valor para cada uno según las probabilidades condicionales.

Esto genera **una realidad concreta**: cada nodo tiene un valor. El agente no sabe estos valores — tiene que descubrirlos observando.

In [ ]:
solver = ExactBayesSolver(world)
true_state = solver.sample_state(seed=EPISODE_SEED)

df_truth = pd.DataFrame([
    {"Nodo": n.name, "Tipo": n.type.value, "Valor": true_state[n.name]}
    for n in world.nodes
])
display(df_truth)

print(f"\nObjetivo: predecir target_outcome = {true_state['target_outcome']}")

## 5. Ver la distribución ANTES de observar nada

Esto es lo que el agente "cree" antes de mirar nada — la distribución prior del target.

Si la prior ya da la respuesta correcta con alta probabilidad, el mundo es demasiado fácil. Si es casi uniforme, hay bastante incertidumbre para reducir.

In [ ]:
prior = solver.posterior("target_outcome")
prior_entropy = solver.entropy(prior)

states = list(prior.keys())
probs = list(prior.values())

state_colors = ["#74C0FC", "#B197FC", "#FF8787", "#69DB7C", "#FFD43B"]
bar_colors = [state_colors[i % len(state_colors)] for i in range(len(states))]
true_idx = states.index(true_state["target_outcome"])
bar_colors[true_idx] = "#e03131"

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(states, probs, color=bar_colors, edgecolor="#343a40", linewidth=0.8, width=0.55, zorder=3)

for bar, p in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.025,
            f"{p:.1%}", ha="center", fontsize=12, fontweight="bold", color="#343a40")

bars[true_idx].set_edgecolor("#c92a2a")
bars[true_idx].set_linewidth(2.5)

ax.set_ylim(0, 1.15)
ax.set_ylabel("Probabilidad", fontsize=12)
ax.set_title("Prior P(target_outcome) — antes de observar nada", fontsize=13, fontweight="bold", pad=12)

ax.text(0.98, 0.95, f"H = {prior_entropy:.2f} bits",
        transform=ax.transAxes, ha="right", va="top",
        fontsize=11, fontweight="bold", color="#e03131",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#fff5f5", edgecolor="#e03131", alpha=0.9))
ax.text(0.98, 0.82, f"Verdad: {true_state['target_outcome']}",
        transform=ax.transAxes, ha="right", va="top",
        fontsize=10, color="#c92a2a", fontweight="bold")

plt.tight_layout()
plt.show()

best_guess = max(prior, key=prior.get)
print(f"Sin observar: diria '{best_guess}' ({prior[best_guess]:.1%})")
print(f"Verdad: '{true_state['target_outcome']}'")
print("Acertaria!" if best_guess == true_state["target_outcome"] else "Se equivocaria — necesita observar.")

## 6. El teacher juega el episodio paso a paso

Ahora el teacher (el jugador perfecto) va a observar variables una por una. En cada turno:
1. Calcula cuánta **información ganaría** observando cada variable disponible
2. Elige la que más información da
3. La observa y actualiza su creencia sobre el target

Vas a ver cómo la distribución cambia turno a turno.

In [ ]:
# Crear episodio y runner
ep_tool = EpisodeGenTool()
episode = ep_tool.generate(world, EpisodeGenConfig(budget=BUDGET, seed=EPISODE_SEED))
runner = EpisodeRunner(world, episode, true_state)

obs_nodes = list(episode.available_nodes)
evidence = {}
history = []

# Prior como paso 0
history.append({
    "step": 0, "label": "Prior",
    "posterior": dict(prior), "entropy": prior_entropy,
    "info_gain": 0, "observed": None, "observed_value": None,
})

print(f"Budget: {BUDGET}  |  Variables: {obs_nodes}  |  Verdad: {true_state['target_outcome']}")
print("=" * 60)

step_rows = []
for step_num in range(BUDGET):
    available = [n for n in obs_nodes if n not in evidence]
    if not available:
        break

    gains = {n: solver.information_gain("target_outcome", evidence, n) for n in available}
    output = solver.optimal_action("target_outcome", evidence, available)
    if output.recommended_action is None:
        print(f"\nTurno {step_num + 1}: entropia ~0, para.")
        break

    node = output.recommended_action.node
    result = runner.step(output.recommended_action)
    evidence[result.observation.node] = result.observation.state

    post = solver.posterior("target_outcome", evidence)
    h = solver.entropy(post)
    map_state = max(post, key=post.get)
    correct = map_state == true_state["target_outcome"]

    step_rows.append({
        "Turno": step_num + 1,
        "Observa": f"{node} = {result.observation.state}",
        "Info Gain": f"{output.information_gain:.4f}",
        "Prediccion": map_state,
        "Correcto": "SI" if correct else "NO",
        "Entropia": f"{h:.3f}",
        "P(target)": {k: f"{v:.1%}" for k, v in post.items()},
    })

    history.append({
        "step": step_num + 1,
        "label": f"Obs: {node}={result.observation.state}",
        "posterior": dict(post), "entropy": h,
        "info_gain": output.information_gain,
        "observed": node, "observed_value": result.observation.state,
    })

# Tabla resumen de pasos
df_steps = pd.DataFrame(step_rows).drop(columns=["P(target)"])
display(df_steps)

# Resultado
final = history[-1]
final_map = max(final["posterior"], key=final["posterior"].get)
print(f"\nPrediccion final: {final_map}  |  Verdad: {true_state['target_outcome']}  |  {'CORRECTO' if final_map == true_state['target_outcome'] else 'INCORRECTO'}")

## 7. Visualizar cómo cambia la creencia turno a turno

Dos gráficos:
- **Arriba**: Cómo cambia la distribución P(target) en cada paso
- **Abajo**: Cómo baja la entropía (incertidumbre) a medida que observa

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9), gridspec_kw={"height_ratios": [3, 1.5]})

# --- Gráfico 1: Evolución de la distribución ---
states = list(history[0]["posterior"].keys())
n_steps = len(history)
x = np.arange(n_steps)
width = 0.22

# Colores por estado
state_palette = ["#74C0FC", "#B197FC", "#FF8787", "#69DB7C", "#FFD43B"]

for i, state in enumerate(states):
    vals = [h["posterior"][state] for h in history]
    c = state_palette[i % len(state_palette)]
    bars = ax1.bar(x + i * width - width * (len(states) - 1) / 2, vals, width,
                   label=state, color=c, edgecolor="#343a40", linewidth=0.5, zorder=3)
    # Valores sobre las barras del último paso
    if n_steps > 0:
        last_val = vals[-1]
        last_bar = bars[-1]
        ax1.text(last_bar.get_x() + last_bar.get_width() / 2, last_val + 0.02,
                 f"{last_val:.0%}", ha="center", fontsize=8, fontweight="bold", color="#343a40")

# Línea de referencia
ax1.axhline(y=1 / len(states), color="#adb5bd", linestyle=":", alpha=0.5, label="Uniforme")

labels = [h["label"] for h in history]
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=25, ha="right", fontsize=9)
ax1.set_ylabel("Probabilidad", fontsize=12)
ax1.set_ylim(0, 1.12)
ax1.set_title(f"Evolución de P(target_outcome) — verdad: {true_state['target_outcome']}",
              fontsize=14, fontweight="bold", pad=12)
ax1.legend(title="Estado", fontsize=10, title_fontsize=10, loc="upper left",
           framealpha=0.9, edgecolor="#dee2e6")

# --- Gráfico 2: Entropía ---
entropies = [h["entropy"] for h in history]
ax2.plot(range(n_steps), entropies, "o-", color="#e03131", linewidth=2.5, markersize=9, zorder=3)
ax2.fill_between(range(n_steps), entropies, alpha=0.12, color="#e03131")

# Etiquetas de valor
for i, ent in enumerate(entropies):
    ax2.text(i, ent + 0.05, f"{ent:.2f}", ha="center", fontsize=9, fontweight="bold", color="#c92a2a")

ax2.set_xticks(range(n_steps))
ax2.set_xticklabels([f"Paso {h['step']}" for h in history], fontsize=9)
ax2.set_ylabel("Entropía (bits)", fontsize=12)
ax2.set_title("Reducción de incertidumbre sobre el target", fontsize=13, fontweight="bold", pad=10)
ax2.set_ylim(bottom=0)

# Badge de reducción total
reduction = entropies[0] - entropies[-1]
ax2.text(0.98, 0.92, f"Reducción: {reduction:.2f} bits ({reduction/entropies[0]*100:.0f}%)",
         transform=ax2.transAxes, ha="right", va="top",
         fontsize=10, fontweight="bold", color="#c92a2a",
         bbox=dict(boxstyle="round,pad=0.4", facecolor="#fff5f5", edgecolor="#e03131", alpha=0.9))

plt.tight_layout()
plt.show()

## 8. Puntuar al agente

Supongamos que un agente (como un LLM) juega el mismo episodio pero observa las variables **en orden aleatorio** en vez de elegir la más informativa. ¿Cómo se compara con el teacher?

In [ ]:
# --- Teacher: ya lo tenemos ---
teacher_final = history[-1]["posterior"]
teacher_evidence = dict(evidence)

# --- Agente random: observa en orden aleatorio ---
rng = np.random.default_rng(42)
random_order = rng.permutation(obs_nodes).tolist()

episode2 = ep_tool.generate(world, EpisodeGenConfig(budget=BUDGET, seed=1))
runner2 = EpisodeRunner(world, episode2, true_state)
random_evidence = {}
random_history = [{"step": 0, "posterior": dict(prior), "entropy": prior_entropy}]

for i, node in enumerate(random_order[:BUDGET]):
    result2 = runner2.step(Action(type=ActionType.OBSERVE, node=node))
    random_evidence[node] = result2.observation.state
    post2 = solver.posterior("target_outcome", random_evidence)
    h2 = solver.entropy(post2)
    random_history.append({"step": i + 1, "posterior": dict(post2), "entropy": h2})

random_final = random_history[-1]["posterior"]

# --- Scores ---
verifier = VerifierTool()
true_post = solver.posterior("target_outcome", {**teacher_evidence})
teacher_score = verifier.score(
    agent_posterior=teacher_final, true_posterior=true_post,
    budget_used=len(teacher_evidence), budget_total=BUDGET,
)
random_true_post = solver.posterior("target_outcome", random_evidence)
random_score = verifier.score(
    agent_posterior=random_final, true_posterior=random_true_post,
    budget_used=len(random_evidence), budget_total=BUDGET,
)

teacher_map = max(teacher_final, key=teacher_final.get)
random_map = max(random_final, key=random_final.get)
truth = true_state["target_outcome"]

df_cmp = pd.DataFrame({
    "Teacher": [teacher_map, truth, "SI" if teacher_map == truth else "NO",
                f"{teacher_score.functional_score:.4f}", f"{history[-1]['entropy']:.3f}"],
    "Random": [random_map, truth, "SI" if random_map == truth else "NO",
               f"{random_score.functional_score:.4f}", f"{random_history[-1]['entropy']:.3f}"],
}, index=["Prediccion", "Verdad", "Correcto?", "KL divergence", "Entropia final"])
display(df_cmp)

print("\n(KL divergence mas bajo = mejor, 0 es perfecto)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

teacher_ent = [h["entropy"] for h in history]
random_ent = [h["entropy"] for h in random_history]

# Teacher
ax.plot(range(len(teacher_ent)), teacher_ent, "o-", color="#2b8a3e", linewidth=2.5,
        markersize=9, label="🧠 Teacher (óptimo)", zorder=3)
ax.fill_between(range(len(teacher_ent)), teacher_ent, alpha=0.08, color="#2b8a3e")

# Random
ax.plot(range(len(random_ent)), random_ent, "s--", color="#e03131", linewidth=2.5,
        markersize=9, label="🎲 Random", zorder=3)
ax.fill_between(range(len(random_ent)), random_ent, alpha=0.08, color="#e03131")

# Etiquetas
for i, (te, re) in enumerate(zip(teacher_ent, random_ent)):
    ax.text(i, te - 0.08, f"{te:.2f}", ha="center", va="top", fontsize=8, color="#2b8a3e", fontweight="bold")
    ax.text(i, re + 0.05, f"{re:.2f}", ha="center", va="bottom", fontsize=8, color="#e03131", fontweight="bold")

ax.set_xlabel("Paso", fontsize=12)
ax.set_ylabel("Entropía (bits) — menor = más seguro", fontsize=12)
ax.set_title("Teacher vs Random: reducción de incertidumbre por paso",
             fontsize=14, fontweight="bold", pad=12)
ax.legend(fontsize=11, framealpha=0.9, edgecolor="#dee2e6", loc="upper right")
ax.set_xticks(range(max(len(teacher_ent), len(random_ent))))
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

## 9. Accuracy del teacher en muchos episodios

¿Qué pasa si corremos muchos mundos y muchos episodios? El teacher debería acertar >90% de las veces.

(Esto tarda ~5 segundos)

In [ ]:
from sreg.models.world import NodeType

n_worlds = 30
n_episodes = 5
correct = 0
total = 0
per_world_acc = []

for w_seed in range(n_worlds):
    w = gen.generate(WorldGenConfig(seed=w_seed, num_nodes=6, edge_strength=EDGE_STRENGTH))
    s = ExactBayesSolver(w)
    obs = [n.name for n in w.nodes if n.type == NodeType.OBSERVABLE]
    w_correct = 0

    for ep_seed in range(n_episodes):
        ts = s.sample_state(seed=w_seed * 1000 + ep_seed)
        _, traj = s.generate_trajectory("target_outcome", obs, len(obs), seed=w_seed * 1000 + ep_seed)
        final_post = traj[-1].posterior
        prediction = max(final_post, key=final_post.get)
        if prediction == ts["target_outcome"]:
            correct += 1
            w_correct += 1
        total += 1

    per_world_acc.append(w_correct / n_episodes)

accuracy = correct / total
print(f"Accuracy del teacher: {accuracy:.1%} ({correct}/{total})")

fig, ax = plt.subplots(figsize=(13, 4.5))
colors = ["#2b8a3e" if a >= 0.8 else "#e8590c" for a in per_world_acc]
ax.bar(range(n_worlds), [a * 100 for a in per_world_acc],
       color=colors, edgecolor="#343a40", linewidth=0.5, zorder=3)

ax.axhline(y=90, color="#e03131", linestyle="--", linewidth=1.5, label="Umbral 90%", zorder=2)
ax.axhline(y=100 / NUM_STATES, color="#adb5bd", linestyle=":", linewidth=1,
           label=f"Azar ({100/NUM_STATES:.0f}%)", zorder=2)

ax.set_xlabel("Mundo (seed)", fontsize=12)
ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_title(f"Accuracy del teacher por mundo ({n_episodes} ep c/u) — Total: {accuracy:.1%}",
             fontsize=14, fontweight="bold", pad=12)
ax.legend(fontsize=10, framealpha=0.9)
ax.set_ylim(0, 108)
ax.set_xticks(range(0, n_worlds, 5))
plt.tight_layout()
plt.show()

## 10. Efecto de edge_strength en la dificultad

¿Qué pasa cuando bajamos `edge_strength`? Las relaciones causales se vuelven más ruidosas, y el teacher acierta menos.

In [ ]:
strengths = [0.2, 0.4, 0.6, 0.8, 1.0]
accs_by_strength = []

for es in strengths:
    correct = 0
    total = 0
    for w_seed in range(20):
        w = gen.generate(WorldGenConfig(seed=w_seed, num_nodes=6, edge_strength=es))
        s = ExactBayesSolver(w)
        obs = [n.name for n in w.nodes if n.type == NodeType.OBSERVABLE]
        for ep_seed in range(5):
            ts = s.sample_state(seed=w_seed * 1000 + ep_seed)
            _, traj = s.generate_trajectory("target_outcome", obs, len(obs), seed=w_seed * 1000 + ep_seed)
            prediction = max(traj[-1].posterior, key=traj[-1].posterior.get)
            if prediction == ts["target_outcome"]:
                correct += 1
            total += 1
    accs_by_strength.append(correct / total)

df_diff = pd.DataFrame({"edge_strength": strengths, "accuracy": [f"{a:.1%}" for a in accs_by_strength]})
display(df_diff)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(strengths, [a * 100 for a in accs_by_strength], "o-", color="#364fc7",
        linewidth=2.5, markersize=11, zorder=3)
ax.fill_between(strengths, [a * 100 for a in accs_by_strength], alpha=0.1, color="#364fc7")

for es, a in zip(strengths, accs_by_strength):
    ax.text(es, a * 100 + 2.5, f"{a:.0%}", ha="center", fontsize=10, fontweight="bold", color="#364fc7")

ax.axhline(y=90, color="#e03131", linestyle="--", linewidth=1.5, alpha=0.7, label="Umbral 90%")
ax.axhline(y=100 / NUM_STATES, color="#adb5bd", linestyle=":", linewidth=1, label=f"Azar ({100/NUM_STATES:.0f}%)")
ax.set_xlabel("Edge Strength (fuerza causal)", fontsize=12)
ax.set_ylabel("Accuracy del teacher (%)", fontsize=12)
ax.set_title("Dificultad controlable: edge_strength bajo = mas dificil",
             fontsize=14, fontweight="bold", pad=12)
ax.legend(fontsize=10, framealpha=0.9, loc="lower right")
ax.set_ylim(0, 108)
ax.set_xlim(0.15, 1.05)
plt.tight_layout()
plt.show()

## 11. Preview del dataset (lo que exportaríamos)

Así se ve una entrada del dataset que generaría Phase 7. Cada fila es un paso de un episodio jugado por el teacher.

In [ ]:
dataset_entry = {
    "world_id": world.id,
    "world_seed": SEED,
    "episode_seed": EPISODE_SEED,
    "template": world.template_family,
    "difficulty": world.difficulty.level,
    "target_node": "target_outcome",
    "true_target_value": true_state["target_outcome"],
    "num_nodes": len(world.nodes),
    "num_observable": sum(1 for n in world.nodes if n.type == NodeType.OBSERVABLE),
    "trajectory": [],
}

for h in history:
    dataset_entry["trajectory"].append({
        "step": h["step"],
        "observed_node": h["observed"],
        "observed_value": h["observed_value"],
        "posterior": {k: round(v, 4) for k, v in h["posterior"].items()},
        "entropy": round(h["entropy"], 4),
        "map_prediction": max(h["posterior"], key=h["posterior"].get),
    })

print("Ejemplo de entrada del dataset (JSONL):\n")
print(json.dumps(dataset_entry, indent=2))

---

## 🧪 Para explorar

Volvé a la celda de parámetros y probá distintas combinaciones:

| Qué cambiar | Ejemplo | Efecto |
|:--|:--|:--|
| `SEED` | `42` | Otro mundo completamente distinto |
| `EDGE_STRENGTH` | `0.3` | Mundo más difícil (relaciones ruidosas) |
| `EDGE_STRENGTH` | `0.95` | Mundo fácil (relaciones casi determinísticas) |
| `NUM_NODES` | `8` | Más variables, más complejo |
| `NUM_STATES` | `2` | Nodos binarios (sí/no) |
| `EPISODE_SEED` | `5` | Misma estructura, distintos "dados" |

> **Tip**: después de cambiar, hacé **Run All** para re-ejecutar todo el notebook.